# Optuna optimization for Raw FC + T1w

**Purpose:** Optimize XGBoost hyperparameters specifically for the raw FC + T1w configuration (6786 features), to enable a fair comparison with the main VAE + T1w pipeline (180 features, separately optimized).

**Key result:** Even with independently optimized XGBoost, raw FC (MAE=6.00) cannot match VAE (MAE=5.62). The results from this notebook are used in the paper and presentation.

## Setup

In [1]:
import sys
from pathlib import Path
import json
import numpy as np
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

_cwd = Path.cwd()
if (_cwd / "src").exists():
    ROOT = _cwd
elif (_cwd.parent / "src").exists():
    ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {_cwd}")

REPO = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Code root:", ROOT)
print("Repo root:", REPO)

/home/nico/Desktop/5to/TesisFinal/.tesis-final-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Code root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Code
Repo root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci


In [2]:
from src.config import Paths, ExperimentConfig
from src.utils_seed import set_global_seed
from src.cohort import build_final_cohort_df
from src.data_io import load_fc_vectors_for_ids
from src.xgb_train import build_feats, clean_xy, train_xgb
from src.metrics import regression_metrics

paths = Paths(
    excel_path=REPO / "Data" / "datos-redlat.xlsx",
    fc_folder=REPO / "Data" / "fc_mats",
    t1w_csv_path=REPO / "Data" / "Redlat_VGM_AAL_.csv",
    out_dir=REPO / "Outputs",
)

cfg = ExperimentConfig(
    seed=42,
    diagnoses_to_use=("CN", "AD", "FTD"),
    test_size=0.10,
    k_folds=5,
    fisher_z=True,
    reuse_artifacts=True,
    use_optuna=False,
)

set_global_seed(cfg.seed)

2026-02-16 22:31:41.385092: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-16 22:31:41.922210: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-16 22:31:43.374074: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Load data (same splits as main pipeline)

In [3]:
cohort_df = build_final_cohort_df(
    paths.excel_path,
    paths.fc_folder,
    paths.t1w_csv_path,
    diagnoses_to_use=cfg.diagnoses_to_use,
)
print(f"Cohort: N={len(cohort_df)}")

Cohort: N=1245


In [4]:
splits_file = paths.out_dir / "splits" / f"splits_seed{cfg.seed}_test{cfg.test_size}.json"
with open(splits_file) as f:
    splits_data = json.load(f)

trainval_ids = splits_data["holdout"]["trainval_ids"]
test_ids = splits_data["holdout"]["test_ids"]
kfold_list = splits_data["folds"]

print(f"Trainval: {len(trainval_ids)}, Test: {len(test_ids)}, Folds: {len(kfold_list)}")

Trainval: 1120, Test: 125, Folds: 5


In [5]:
X_all = load_fc_vectors_for_ids(
    paths.fc_folder,
    cohort_df["record_id"].tolist(),
    apply_fisher_z=cfg.fisher_z,
)

id_to_idx = {rec_id: i for i, rec_id in enumerate(cohort_df["record_id"])}
trainval_idx = [id_to_idx[rid] for rid in trainval_ids]
test_idx = [id_to_idx[rid] for rid in test_ids]

X_trainval = X_all[trainval_idx]
X_test = X_all[test_idx]

print(f"FC — trainval: {X_trainval.shape}, test: {X_test.shape}")

FC — trainval: (1120, 6670), test: (125, 6670)


In [6]:
def get_data_for_ids(ids):
    df_sub = cohort_df[cohort_df["record_id"].isin(ids)].set_index("record_id").loc[ids].reset_index()
    y = df_sub["age"].values
    T1 = df_sub[[c for c in df_sub.columns if c.startswith("t1_")]].values
    return y, T1

y_tr, T1_tr = get_data_for_ids(trainval_ids)
y_te, T1_te = get_data_for_ids(test_ids)

print(f"y_tr: {y_tr.shape}, T1_tr: {T1_tr.shape}")
print(f"y_te: {y_te.shape}, T1_te: {T1_te.shape}")

y_tr: (1120,), T1_tr: (1120, 116)
y_te: (125,), T1_te: (125, 116)


## Build Raw FC + T1w features

In [7]:
X_raw_tr = build_feats(Z=X_trainval, T1=T1_tr)
X_raw_te = build_feats(Z=X_test, T1=T1_te)

X_raw_tr, y_tr_clean = clean_xy(X_raw_tr, y_tr)
X_raw_te, y_te_clean = clean_xy(X_raw_te, y_te)

print(f"Raw FC + T1w — train: {X_raw_tr.shape}, test: {X_raw_te.shape}")
print(f"Features: {X_raw_tr.shape[1]} (6670 FC + 116 T1w)")

Raw FC + T1w — train: (1120, 6786), test: (125, 6786)
Features: 6786 (6670 FC + 116 T1w)


## Optuna optimization (50 trials, 5-fold CV)

Search space adapted for 6786 features (38x more than the 180-feature pipeline):
- **max_depth 2-5** (deeper trees are extremely slow and overfit with this many features)
- **colsample_bytree 0.05-0.3** (each tree sees 5-30% of features -- critical for speed and regularization)
- **n_estimators 100-600** (fewer, shallower trees)
- **Stronger regularization** (reg_alpha up to 10, reg_lambda up to 100)

Each trial should take ~30s-2min. Total estimated: ~30-60 min.

In [8]:
SEED = 42
N_TRIALS = 50
K_FOLDS = 5

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
sampler = optuna.samplers.TPESampler(seed=SEED)

def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 0.3),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 100.0, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 20.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "tree_method": "hist",
        "random_state": SEED,
        "eval_metric": "mae",
        "n_jobs": -1,
    }

    maes = []
    for tr_idx, va_idx in kf.split(X_raw_tr):
        model = XGBRegressor(**params)
        model.fit(X_raw_tr[tr_idx], y_tr_clean[tr_idx], verbose=False)
        pred = model.predict(X_raw_tr[va_idx])
        maes.append(mean_absolute_error(y_tr_clean[va_idx], pred))
    return float(np.mean(maes))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest CV MAE: {study.best_value:.4f}")
print(f"Best params: {study.best_trial.params}")

Best trial: 43. Best value: 6.53247: 100%|██████████| 50/50 [1:56:56<00:00, 140.32s/it]


Best CV MAE: 6.5325
Best params: {'n_estimators': 458, 'max_depth': 3, 'learning_rate': 0.03530771088566478, 'subsample': 0.9559350035129378, 'colsample_bytree': 0.10442179408200097, 'reg_alpha': 0.07562391855194042, 'reg_lambda': 0.19912112625989395, 'min_child_weight': 10.287060507653141, 'gamma': 2.493016501874793}


## Evaluate optimized model on test set

In [9]:
best_params_raw = study.best_trial.params
best_params_raw.update({
    "tree_method": "hist",
    "random_state": SEED,
    "eval_metric": "mae",
    "n_jobs": -1,
})

model_raw_opt = train_xgb(X_raw_tr, y_tr_clean, best_params_raw)
y_pred_raw_opt = model_raw_opt.predict(X_raw_te)
metrics_raw_opt = regression_metrics(y_te_clean, y_pred_raw_opt)

print("=== Optimized Raw FC + T1w (test set) ===")
for k, v in metrics_raw_opt.items():
    print(f"  {k}: {v:.4f}")

=== Optimized Raw FC + T1w (test set) ===
  MAE: 5.9965
  RMSE: 7.5475
  R2: 0.3535
  Pearson: 0.6012


## Compare: shared params vs optimized params vs VAE pipeline

In [10]:
# Load main pipeline best result (VAE+T1w with its own Optuna-optimized XGBoost)
abl_path = paths.out_dir / "metrics" / "ablation_results.json"
with open(abl_path) as f:
    ablation = json.load(f)
m_vae_opt = ablation["VAE(Z) + T1w"]

# Definitive comparison: each configuration with its own Optuna-optimized params
print("=" * 80)
print("FAIR COMPARISON: Raw FC + T1w vs VAE + T1w (each Optuna-optimized)")
print("=" * 80)
print(f"{'Configuration':<35} | {'Features':>8} | {'MAE':>6} | {'RMSE':>6} | {'R²':>6} | {'Pearson':>7}")
print("-" * 80)
print(f"{'Raw FC+T1w (own Optuna)':<35} | {6786:>8} | {metrics_raw_opt['MAE']:6.2f} | {metrics_raw_opt['RMSE']:6.2f} | {metrics_raw_opt['R2']:6.3f} | {metrics_raw_opt['Pearson']:7.3f}")
print(f"{'VAE+T1w (own Optuna)':<35} | {180:>8} | {m_vae_opt['MAE']:6.2f} | {m_vae_opt['RMSE']:6.2f} | {m_vae_opt['R2']:6.3f} | {m_vae_opt['Pearson']:7.3f}")
print("-" * 80)
mae_diff = metrics_raw_opt['MAE'] - m_vae_opt['MAE']
r2_diff = m_vae_opt['R2'] - metrics_raw_opt['R2']
print(f"VAE advantage:  {mae_diff:+.2f} years MAE,  {r2_diff:+.3f} R²")
print(f"Feature reduction: 6786 → 180 ({(1 - 180/6786)*100:.1f}% reduction)")
print("=" * 80)
print("\nConclusion: VAE wins on every metric even with independently optimized XGBoost.")
print("These are the numbers used in the paper and presentation.")

COMPARISON: Raw FC + T1w vs VAE + T1w
Configuration                       | Features |    MAE |   RMSE |     R² | Pearson
-------------------------------------------------------------------------------------
Raw FC+T1w (shared params)          |     6786 |   6.49 |   8.40 |  0.198 |   0.488
Raw FC+T1w (Optuna-optimized)       |     6786 |   6.00 |   7.55 |  0.354 |   0.601
-------------------------------------------------------------------------------------
VAE+T1w (shared params)             |      180 |   6.48 |   7.98 |  0.277 |   0.543
VAE+T1w (Optuna-optimized)          |      180 |   5.62 |   7.20 |  0.411 |   0.644

--- Key comparisons ---
Shared params improvement (Optuna):  MAE +0.49,  R² +0.155
Raw optimized vs VAE optimized:      MAE +0.38,  R² +0.058


## Compare hyperparameters

In [11]:
# Load the VAE+T1w optimized params
with open(paths.out_dir / "optuna" / "xgb_best_cv.json") as f:
    vae_params = json.load(f)["best_params"]

print(f"{'Parameter':<20} | {'VAE+T1w (180 feats)':>22} | {'Raw FC+T1w (6786 feats)':>24}")
print("-" * 72)
for key in sorted(set(list(vae_params.keys()) + list(best_params_raw.keys()))):
    v1 = vae_params.get(key, '-')
    v2 = best_params_raw.get(key, '-')
    if isinstance(v1, float):
        v1 = f"{v1:.6f}"
    if isinstance(v2, float):
        v2 = f"{v2:.6f}"
    print(f"{key:<20} | {str(v1):>22} | {str(v2):>24}")

Parameter            |    VAE+T1w (180 feats) |  Raw FC+T1w (6786 feats)
------------------------------------------------------------------------
colsample_bytree     |               0.950597 |                 0.104422
eval_metric          |                    mae |                      mae
gamma                |               0.674946 |                 2.493017
learning_rate        |               0.011042 |                 0.035308
max_depth            |                      6 |                        3
min_child_weight     |               1.141535 |                10.287061
n_estimators         |                   2103 |                      458
n_jobs               |                      - |                       -1
random_state         |                     42 |                       42
reg_alpha            |               0.001355 |                 0.075624
reg_lambda           |               7.084194 |                 0.199121
subsample            |               0.502133 |    

## Save results

In [ ]:
optuna_results = {
    "raw_fc_t1w_optimized": {
        "MAE": float(metrics_raw_opt["MAE"]),
        "RMSE": float(metrics_raw_opt["RMSE"]),
        "R2": float(metrics_raw_opt["R2"]),
        "Pearson": float(metrics_raw_opt["Pearson"]),
    },
    "vae_t1w_optimized": {
        "MAE": float(m_vae_opt["MAE"]),
        "RMSE": float(m_vae_opt["RMSE"]),
        "R2": float(m_vae_opt["R2"]),
        "Pearson": float(m_vae_opt["Pearson"]),
    },
    "raw_fc_best_params": best_params_raw,
    "raw_fc_best_cv_mae": float(study.best_value),
}

save_path = paths.out_dir / "experiments" / "exp3_optuna_raw_fc.json"
with open(save_path, "w") as f:
    json.dump(optuna_results, f, indent=2)

print(f"Results saved: {save_path}")